# fudosan-library JupyterLab Intro

This notebook shows how to interact with the MLIT Real Estate Information Library API using the synchronous `FudosanClient`.

## 0. Prerequisites

Before running the cells, make sure you have:

- Installed the package (optionally with geo extras):
  ```bash
  uv pip install -e ..[dev,geo]
  ```
- Exported your API key:
  ```bash
  export FUDOSAN_API_KEY="your-api-key"
  ```

In [ ]:
import os
from fudosan_library import FudosanClient

API_KEY = os.environ.get("FUDOSAN_API_KEY")
if not API_KEY:
    raise RuntimeError("Set the FUDOSAN_API_KEY environment variable before running this notebook.")

client = FudosanClient(api_key=API_KEY)
client

## 1. Fetch trade price records (XIT001)

Supply the municipality code, year, and price classification to retrieve transaction data.

In [ ]:
records = client.get_trade_prices(
    year=2023,
    price_classification='01',
    area='13',
    city='13101',
)

len(records), records[0]

In [ ]:
data_rows = [record.model_dump(mode="python") for record in records]

try:
    import pandas as pd
except ModuleNotFoundError:
    data_rows[:3]
else:
    pd.DataFrame(data_rows).head()

## 2. Retrieve trade points (XPT001) and view as GeoDataFrame

If you installed the geo extras, you can convert the GeoJSON response into a `GeoDataFrame` for mapping or spatial analysis.

In [ ]:
trade_points = client.get_trade_points(
    response_format='geojson',
    z=13,
    x=7450,
    y=3220,
    period_from='20231',
    period_to='20234',
    price_classification='01',
)

from fudosan_library.geo import geojson_to_gdf

gdf = geojson_to_gdf(trade_points)
gdf.head()

## 3. Explore land-use zones (XKT002)

The use zone endpoint also returns GeoJSON. You can re-use the same helper to inspect it.

In [ ]:
use_zone = client.get_land_use_zones(
    response_format='geojson',
    z=12,
    x=2100,
    y=1040,
)
use_zone['features'][0]